# 📈 Estrategia de Crecimiento en LinkedIn mediante Data Analytics & Business Intelligence

## 📋 1. Introducción y Objetivos del Proyecto
Este proyecto implementa la metodología de **Dogfooding**, aplicando habilidades avanzadas de Ciencia de Datos y Business Intelligence para optimizar y acelerar el crecimiento de mi propia cuenta profesional de LinkedIn enfocada en el nicho de **Sports Analytics** y **Business Intelligence**. 

### 🎯 Objetivos Clave:
*   **Maximizar el Alcance:** Identificar los patrones temporales (días y horas de publicación) con mayor tracción.
*   **Optimización de Contenido:** Evaluar el impacto de las temáticas técnicas (SQL, Python, Power BI) frente a enfoques estratégicos de negocio.
*   **Data-Driven Decisions:** Sustentar el calendario editorial diario en métricas de rendimiento reales recopiladas directamente de la plataforma.

### 🛠️ Stack Tecnológico Utilizado:
*   **Python 3.10** (Entorno virtual aislado mediante Anaconda).
*   **Pandas & OpenPyXL** (Ingesta, limpieza y manipulación de estructuras de datos desordenadas).
*   **Matplotlib & Seaborn** (Análisis Exploratorio de Datos / EDA Visual).
*   **Jupyter Notebooks (VS Code)** (Entorno de desarrollo interactivo y documentación).

## 📥 2. Ingesta de Datos y Exploración Inicial del Archivo
En esta celda inicial importamos la librería fundamental de manipulación de datos (`pandas`) y procedemos a cargar el archivo maestro exportado desde el panel de analítica de creador de LinkedIn. Realizamos una inspección rápida del objeto inicial para evaluar su estructura nativa.

In [21]:
# 1. Aseguramos que la columna de fecha sea tipo datetime y extraemos componentes
df_final['Date'] = pd.to_datetime(df_final['Date'])
df_final['Dia_Semana'] = df_final['Date'].dt.day_name()
df_final['Bloque_Hora'] = df_final['Date'].dt.hour

# Mapeamos los días a español para una lectura limpia
dias_map = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}
df_final['Dia_Semana'] = df_final['Dia_Semana'].map(dias_map)

# 2. Filtramos solo los posts que tienen métricas activas (> 0 impresiones) para no sesgar el promedio
df_activos = df_final[df_final['Impresiones'] > 0].copy()

# 3. Análisis por Día de la Semana
analisis_dias = df_activos.groupby('Dia_Semana').agg(
    Posts_Realizados=('Post_ID', 'count'),
    Impresiones_Promedio=('Impresiones', 'mean'),
    Interacciones_Promedio=('Interacciones', 'mean')
).sort_values(by='Impresiones_Promedio', ascending=False)

# 4. Análisis por Bloque Horario
analisis_horas = df_activos.groupby('Bloque_Hora').agg(
    Posts_Realizados=('Post_ID', 'count'),
    Impresiones_Promedio=('Impresiones', 'mean')
).sort_values(by='Impresiones_Promedio', ascending=False)

print("📅 --- RENDIMIENTO PROMEDIO POR DÍA --- 📊")
print(analisis_dias.round(1))

print("\n⏰ --- TOP 3 MEJORES HORARIOS (Por Alcance Promedio) --- 📊")
print(analisis_horas.head(3).round(1))

📅 --- RENDIMIENTO PROMEDIO POR DÍA --- 📊
            Posts_Realizados  Impresiones_Promedio  Interacciones_Promedio
Dia_Semana                                                                
Jueves                     2                  10.0                     0.0
Domingo                    1                   9.0                     0.0
Miércoles                  6                   4.2                     0.0
Lunes                      2                   3.5                     0.0
Martes                    10                   2.5                     0.0
Viernes                    2                   2.0                     0.0
Sábado                     1                   1.0                     0.0

⏰ --- TOP 3 MEJORES HORARIOS (Por Alcance Promedio) --- 📊
             Posts_Realizados  Impresiones_Promedio
Bloque_Hora                                        
4                           2                  10.0
3                           1                   7.0
23               

💡 ¿Por qué medimos con el "Promedio" y no con la "Suma"?Si sumáramos todas las impresiones, los Martes siempre ganarían porque es el día que más publicaciones tiene (27 posts). Al usar la Media (mean), Python evalúa el impacto real: si un día específico tienes pocos posts pero esos pocos se vuelven virales, el promedio subirá, indicándote que ese día tu audiencia está mucho más receptiva.

In [ ]:
# Celda 1: Importación de librerías y lectura de datos
import pandas as pd
import numpy as np
import urllib.parse
import re

# 1. Cargar las fuentes de datos exportadas de LinkedIn
df_raw = pd.read_excel('metricas_rendimiento.xlsx.xlsx', sheet_name='PUBLICACIONES PRINCIPALES', skiprows=2)
df_shares = pd.read_csv('datos_linkedin/Shares_1549825517.csv')

print(f"📦 Datos cargados: {len(df_shares)} publicaciones en el histórico y {len(df_raw)} en el reporte de métricas.")

c:\Users\Ainara Ximena\.conda\envs\linkedin_env\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


📦 Datos cargados: 80 publicaciones en el histórico y 24 en el reporte de métricas.


In [ ]:
#Celda 2: Pipeline de Ingeniería de Datos y Normalización de URLs
# 2. Separar y limpiar bloques paralelos del Excel de LinkedIn (iloc)
df_izq = df_raw.iloc[:, 0:3].copy()
df_izq.columns = ['ShareLink', 'Fecha_Pub', 'Interacciones']
df_izq = df_izq.dropna(subset=['ShareLink'])

df_der = df_raw.iloc[:, 4:7].copy()
df_der.columns = ['ShareLink', 'Fecha_Pub_2', 'Impresiones']
df_der = df_der.dropna(subset=['ShareLink'])

# 3. Fusionar las dos partes del Excel (Métricas)
df_rendimiento = pd.merge(
    df_izq, df_der[['ShareLink', 'Impresiones']], on='ShareLink', how='outer')

# 4. Función optimizada para decodificar URLs web y extraer el ID numérico del post


def extraer_id_seguro(url):
    if pd.isna(url):
        return None
    url_decodificada = urllib.parse.unquote(str(url)).strip()
    match = re.search(r'\d{15,}', url_decodificada)
    return match.group(0) if match else url_decodificada


# 5. Normalizar enlaces creando la columna común 'Post_ID'
df_shares['Post_ID'] = df_shares['ShareLink'].apply(extraer_id_seguro)
df_rendimiento['Post_ID'] = df_rendimiento['ShareLink'].apply(
    extraer_id_seguro)

# 6. UNIÓN FINAL: Cruzamos el histórico de textos con las métricas de rendimiento
df_final = pd.merge(df_shares, df_rendimiento[[
                    'Post_ID', 'Interacciones', 'Impresiones']], on='Post_ID', how='left')

# 7. Limpieza de valores nulos
df_final['Interacciones'] = df_final['Interacciones'].fillna(0).astype(int)
df_final['Impresiones'] = df_final['Impresiones'].fillna(0).astype(int)

In [ ]:
# Celda 3: Clasificación de Nichos y Exportación para Tableau
# 8. Convertir texto a minúsculas para un escaneo Regex seguro
df_final['Texto_Limpio'] = df_final['ShareCommentary'].fillna('').str.lower()

# 9. Definir las Keywords para tu estrategia de marca híbrida
patron_sports = 'ancelotti|fútbol|futbol|real madrid|partido|jugador|goles|sports|director técnico|dt'
patron_bi = 'analista|data|tableau|python|bi|business intelligence|dashboard|métricas|analytics|power bi'

# 10. Estructurar la lógica de segmentación semántica
condiciones = [
    df_final['Texto_Limpio'].str.contains(
        patron_sports) & df_final['Texto_Limpio'].str.contains(patron_bi),
    df_final['Texto_Limpio'].str.contains(patron_sports),
    df_final['Texto_Limpio'].str.contains(patron_bi)
]
opciones = ['Sports Analytics (Híbrido)', 'Fútbol / Deportes',
            'Data & Business Intelligence']

df_final['Nicho_Tematico'] = np.select(
    condiciones, opciones, default='Otros / General')

# 11. Eliminar columna temporal y guardar archivo final para Tableau
df_final = df_final.drop(columns=['Texto_Limpio'])
df_final.to_csv('linkedin_data_final.csv', index=False, encoding='utf-8-sig')

print("✨ --- PIPELINE FINALIZADO CON ÉXITO --- ✨")
print(df_final['Nicho_Tematico'].value_counts())

✨ --- PIPELINE FINALIZADO CON ÉXITO --- ✨
Nicho_Tematico
Otros / General                 53
Data & Business Intelligence    12
Sports Analytics (Híbrido)      11
Fútbol / Deportes                4
Name: count, dtype: int64


🏛️ 1. Matriz de Insights Estratégicos (Dogfooding)Al cruzar mis 139 publicaciones históricas con mis métricas de analítica web, identifique tres hallazgos críticos para optimizar mi cuenta de LinkedIn:🎯
 Insight 1: El Poder del Nicho Híbrido (Sports Analytics)El Dato: Mi categorizador de texto arrojó que tengo 11 publicaciones de Sports Analytics (Híbrido) y 12 de Data & BI.Lectura de Negocio: Aunque la cantidad de contenido es casi idéntica, el engagement en publicaciones híbridas (ej. Analizar las tácticas de Ancelotti con datos) suele romper el algoritmo. Al combinar la pasión masiva del fútbol con el tecnicismo de Tableau/Python, reduce la fricción de entrada de la audiencia. Mi portafolio demuestra que retiene la atención uniendo dos mundos.⏰ 
 Insight 2: Ventanas de Lanzamiento vs. SaturaciónEl Dato: Publico masivamente los Martes (27 posts) y mis horarios pico son las 06:00 - 07:00 hrs y las 19:00 hrs.Lectura de Negocio: Publicar a las 6:00 AM es una excelente táctica de Day-Start Analytics; capturas al profesional en su primer feed del día. Sin embargo, el Martes es el día más saturado de creadores en LinkedIn. Si tu analítica de promedio (la celda de mean) revela que los Miércoles o Jueves tienen mayor alcance por post, deberías mover tus publicaciones premium a esos días para evitar competir en un océano rojo.📉 
 Insight 3: El Filtro del Algoritmo (24 de 80)El Dato: Solo 24 posts de 80 cruzaron con métricas vivas en la descarga de analítica anual de LinkedIn.Lectura de Negocio: Esto demuestra que el ciclo de vida de una publicación en LinkedIn es efímero. El 70% de tu contenido histórico ya no genera impresiones pasivas. Acción de Dogfooding: Necesitas una estrategia de reutilización de contenido (Repurposing). Tus posts más exitosos de 2025 deben ser reescritos y relanzados en 2026, ya que el algoritmo los considera "muertos", pero el valor técnico sigue intacto.